# Exponential integrators in NGSolve — Part 4: the nonlinear Allen–Cahn equation

$$
\partial_t u = \varepsilon\,\Delta u + u - u^3 \quad\text{in } (0,1)^2, \qquad \partial_n u = 0,
$$

is the $L^2$-gradient flow of $\mathcal E(u) = \int \tfrac{\varepsilon}{2}|\nabla u|^2 + \tfrac14(u^2-1)^2$: phases $u \approx \pm1$ form, interfaces of width $\sim\sqrt\varepsilon$ move by curvature, the energy decreases. Semi-discrete form: $M\dot u = -K_\varepsilon u + F(u)$, $F(u)_i = \int (u-u^3)\phi_i$, with a stiff linear part $A = M^{-1}K_\varepsilon$ and a non-stiff nonlinearity $g(u) = M^{-1}F(u)$.

* **Exponential Euler**: $u^{n+1} = e^{-\Delta tA}u^n + \Delta t\,\varphi_1(-\Delta tA)\,g(u^n)$, order 1, one factorisation for the whole run.
* **Exponential Rosenbrock–Euler** (Hochbruck–Ostermann–Schweitzer 2009): linearise at $u^n$ with $J_n = -A + M^{-1}F'(u^n)$ and take $u^{n+1} = u^n + \Delta t\,\varphi_1(\Delta tJ_n)\big(-Au^n + g(u^n)\big)$, order 2, one factorisation per step (like a Newton step).
* **IMEX Euler**: $(M + \Delta tK_\varepsilon)u^{n+1} = Mu^n + \Delta t\,F(u^n)$, order 1.

In [ ]:
import sys; sys.path.insert(0, "..")
from ngsolve import *
from ngsolve.webgui import Draw
import numpy as np, matplotlib.pyplot as plt
from time import time as walltime
from expint_tools import phi, KrylovExp
ngsglobals.msg_level = 0

In [ ]:
eps = 0.005
def setup(maxh, order=2):
    mesh = Mesh(unit_square.GenerateMesh(maxh=maxh))
    fes = H1(mesh, order=order)                                  # Neumann: no Dirichlet dofs
    u, v = fes.TnT()
    K = BilinearForm(eps*grad(u)*grad(v)*dx, symmetric=False).Assemble().mat
    M = BilinearForm(u*v*dx, symmetric=False).Assemble().mat
    gfu = GridFunction(fes)
    F = LinearForm(fes); F += (gfu - gfu**3)*v*dx               # nonlinearity F(u), depends on gfu
    return dict(mesh=mesh, fes=fes, u=u, v=v, K=K, M=M, gfu=gfu, F=F, freedofs=fes.FreeDofs(),
                invM=M.Inverse(fes.FreeDofs(), inverse="sparsecholesky"))

u0 = 0.3*cos(3*pi*x)*cos(2*pi*y) + 0.2*sin(5*pi*x)*sin(4*pi*y) + 0.1*cos(7*pi*x)
def energy(P):
    return Integrate(eps/2*grad(P["gfu"])**2 + 0.25*(P["gfu"]**2 - 1)**2, P["mesh"])

P = setup(0.04)
P["gfu"].Set(u0); print("ndof =", P["fes"].ndof, "  initial energy =", energy(P))
Draw(P["gfu"], P["mesh"], "u0", min=-1, max=1);

## The three time steppers

`KrylovExp` does the work. For exponential Rosenbrock–Euler it is re-created every step with the Jacobian $K_\varepsilon - M_{F'(u^n)}$, $F'(u)_{ij} = \int (1-3u^2)\phi_j\phi_i$, assembled with the current `gfu` as coefficient: $\varphi_1(\Delta tJ_n)$ is exactly `KrylovExp(M, jac, dt).Phi(1, ·)`.

In [ ]:
def exp_euler(P, dt, Tend, tol=1e-8, callback=None):
    M, K, F, gfu, invM = P["M"], P["K"], P["F"], P["gfu"], P["invM"]
    E = KrylovExp(M, K, dt, P["freedofs"], tol=tol)             # one factorisation for the whole run
    for n in range(int(round(Tend/dt))):
        F.Assemble()
        gfu.vec.data = E*gfu.vec + dt*E.Phi(1, invM*F.vec)
        if callback: callback(n)

def exp_rosenbrock_euler(P, dt, Tend, tol=1e-8, callback=None):
    M, K, F, gfu, invM, u, v = P["M"], P["K"], P["F"], P["gfu"], P["invM"], P["u"], P["v"]
    for n in range(int(round(Tend/dt))):
        F.Assemble()
        jac = BilinearForm(eps*grad(u)*grad(v)*dx - (1 - 3*gfu**2)*u*v*dx, symmetric=False).Assemble().mat
        E = KrylovExp(M, jac, dt, P["freedofs"], tol=tol)      # new factorisation each step
        rhs = M.CreateColVector(); rhs.data = F.vec - K*gfu.vec                # M u' = F(u) - K u
        gfu.vec.data += dt*E.Phi(1, invM*rhs)
        if callback: callback(n)

def imex_euler(P, dt, Tend, callback=None):
    M, K, F, gfu = P["M"], P["K"], P["F"], P["gfu"]
    mstar = M.CreateMatrix(); mstar.AsVector().data = M.AsVector() + dt*K.AsVector()
    inv = mstar.Inverse(P["freedofs"], inverse="sparsecholesky")
    for n in range(int(round(Tend/dt))):
        F.Assemble()
        gfu.vec.data = inv*(M*gfu.vec + dt*F.vec)
        if callback: callback(n)

## Phase separation and energy decay

$T = 4$, $\Delta t = 0.05$. The exponential methods stay stable and accurate for much larger steps than the IMEX scheme (see the table at the end).

In [ ]:
Tend, dt = 4.0, 0.05
runs = {"exponential Euler": exp_euler, "exponential Rosenbrock-Euler": exp_rosenbrock_euler, "IMEX Euler": imex_euler}
energies, times = {}, {}
for name, method in runs.items():
    P["gfu"].Set(u0); en = [energy(P)]
    t0 = walltime(); method(P, dt, Tend, callback=lambda n: en.append(energy(P))); times[name] = walltime() - t0
    energies[name] = en
    print(f"{name:30s}  {times[name]:6.2f} s   final energy {en[-1]:.6f}")
plt.figure(figsize=(6, 3.5))
for name, en in energies.items(): plt.plot(np.arange(len(en))*dt, en, label=name)
plt.xlabel("t"); plt.ylabel("energy"); plt.legend(); plt.grid(); plt.title(f"Allen-Cahn, dt = {dt}"); plt.show()

In [ ]:
# animation of the exponential Rosenbrock-Euler solution
P["gfu"].Set(u0)
gfut = GridFunction(P["fes"], multidim=0); gfut.AddMultiDimComponent(P["gfu"].vec)
exp_rosenbrock_euler(P, dt, Tend, callback=lambda n: gfut.AddMultiDimComponent(P["gfu"].vec) if (n+1) % 4 == 0 else None)
Draw(gfut, P["mesh"], interpolate_multidim=True, animate=True, min=-1, max=1);

## Convergence in $\Delta t$

Reference: exponential Rosenbrock–Euler with a very small step on a coarser mesh; errors in the $M$-norm at $T = 0.4$, during the fast initial phase separation.

In [ ]:
Q = setup(0.1)
Tend = 0.4
def run(method, dt):
    Q["gfu"].Set(u0); method(Q, dt, Tend, **({} if method is imex_euler else {"tol": 1e-10}))
    return Q["gfu"].vec.FV().NumPy().copy()
refsol = run(exp_rosenbrock_euler, 0.0025)
Mq = Q["M"]; tmpv = Mq.CreateColVector()
def err_M(w):
    tmpv.FV().NumPy()[:] = w - refsol; return sqrt(InnerProduct(tmpv, Mq*tmpv))

dts = [0.1, 0.05, 0.025, 0.0125]
err = {name: [err_M(run(method, dt)) for dt in dts] for name, method in runs.items()}
print(f"{'dt':>8s} " + " ".join(f"{k:>30s}" for k in err))
for i, dt_ in enumerate(dts): print(f"{dt_:8.4f} " + " ".join(f"{err[k][i]:30.2e}" for k in err))
orders = {k: float(np.log2(e[-2]/e[-1])) for k, e in err.items()}
print("estimated orders:", {k: round(o, 2) for k, o in orders.items()})
plt.figure(figsize=(5.5, 3.8))
for k, e in err.items(): plt.loglog(dts, e, "o-", label=k)
plt.loglog(dts, [d/dts[0]*err["IMEX Euler"][0] for d in dts], "k:", lw=.8, label=r"$O(\Delta t)$")
plt.loglog(dts, [(d/dts[0])**2*err["exponential Rosenbrock-Euler"][0] for d in dts], "k--", lw=.8, label=r"$O(\Delta t^2)$")
plt.xlabel(r"$\Delta t$"); plt.ylabel("error at T = 0.4"); plt.grid(); plt.legend(); plt.show()

## Large time steps

Final energy after $T = 4$: the step size of the exponential methods is limited only by the accuracy of the nonlinearity.

In [ ]:
Tend = 4.0
print(f"{'dt':>6s} " + " ".join(f"{k:>30s}" for k in runs))
for dt_ in [0.5, 0.25, 0.1, 0.05]:
    row = []
    for name, method in runs.items():
        P["gfu"].Set(u0); method(P, dt_, Tend); row.append(energy(P))
    print(f"{dt_:6.2f} " + " ".join(f"{e:30.5f}" for e in row))

In [ ]:
# tests
assert 0.8 < orders["exponential Euler"] < 1.3 and 0.8 < orders["IMEX Euler"] < 1.3
assert 1.7 < orders["exponential Rosenbrock-Euler"] < 2.4
assert err["exponential Rosenbrock-Euler"][-1] < err["exponential Euler"][-1] < err["IMEX Euler"][-1]
for name, en in energies.items():
    assert all(np.diff(en) <= 1e-8), name          # monotone energy decay
print("all tests passed")

## Summary

* Semilinear problems: stiff linear part exact ($e^{-\Delta tA}$, $\varphi_1$), nonlinearity explicit — exponential Euler, order 1, one factorisation per run.
* Linearising every step (exponential Rosenbrock–Euler) gives order 2 for one factorisation per step; higher-order exponential Rosenbrock and Runge–Kutta methods follow the same pattern with more $\varphi$-functions.

**References:** M. Hochbruck, A. Ostermann, J. Schweitzer, SIAM J. Numer. Anal. 47 (2009); M. Hochbruck, A. Ostermann, Acta Numerica 19 (2010); S. M. Cox, P. C. Matthews, J. Comput. Phys. 176 (2002); A.-K. Kassam, L. N. Trefethen, SIAM J. Sci. Comput. 26 (2005).